# Логирование Градиентного бустинга в MLflow для проекта "Определение популярности геолокации для размещения банкомата"

## 0) Установка зависимостей
Если запускаете впервые, раскомментируйте строку ниже и выполните ячейку.

## 1) Импорты и настройки окружения

In [24]:
import os
import warnings

import pandas as pd
import numpy as np
from dotenv import load_dotenv

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import learning_curve

import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

warnings.filterwarnings("ignore")
load_dotenv()

True

In [11]:
# Для локального стенда из docker-compose
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID", "admin")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY", "password")
raw_s3_endpoint = os.getenv("MLFLOW_S3_ENDPOINT_URL", "http://localhost:9000")

# `minio` резолвится только внутри docker-сети. Для локального ноутбука нужен localhost.
if "minio:9000" in raw_s3_endpoint:
    raw_s3_endpoint = "http://localhost:9000"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = raw_s3_endpoint

# Если на сервере включена basic-auth, задайте логин/пароль
# os.environ["MLFLOW_TRACKING_USERNAME"] = os.getenv("MLFLOW_TRACKING_USERNAME", "admin")
# os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("MLFLOW_TRACKING_PASSWORD", "password")

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5050")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print("MLflow URI:", mlflow.get_tracking_uri())

MLflow URI: http://localhost:5050


In [12]:
# Быстрая проверка подключения
mlflow.search_experiments(max_results=3)

[<Experiment: artifact_location='s3://mlflow-bucket/mlflow', creation_time=1780079042525, experiment_id='1', last_update_time=1780079042525, lifecycle_stage='active', name='gradient_boosting_regression', tags={}>,
 <Experiment: artifact_location='s3://mlflow-bucket/mlflow/0', creation_time=1780078919701, experiment_id='0', last_update_time=1780078919701, lifecycle_stage='active', name='Default', tags={}>]

## 2) Подготовка датасета

In [13]:
df = pd.read_csv('./data/train_with_new_features.csv')
df = df.drop(columns=['id', 'address', 'address_rus'])
df = df.dropna()

df['population'] = df['population'].str.replace("\xa0", "").astype(float)
df['atm_group'] = df['atm_group'].astype('float')

X = df.drop(columns=['target'])
y = df['target']

# Загрузка тестовых данных
df_test = pd.read_csv('./data/test_with_new_features.csv')
df_test = df_test.drop(columns=['Unnamed: 0', 'id', 'address', 'address_rus'])
df_test = df_test.dropna()
df_test['population'] = df_test['population'].astype(str).str.replace("\xa0", "").astype(float)
df_test['atm_group'] = df_test['atm_group'].astype('float')

X_test = df_test.drop(columns=['target'])
y_test = df_test['target']

# Конвертируем колонки, которые должны быть int
int_columns = ['schools_nearby', 'supermarket_nearby', 'mall_nearby', 'bar_nearby', 
               'cafe_nearby', 'restaurant_nearby', 'police_nearby', 'post_office_nearby',
               'place_of_worship_nearby', 'university_nearby', 'cinema_nearby', 
               'casino_nearby', 'nightclub_nearby']

for col in int_columns:
    X_test[col] = X_test[col].astype('int64')

# Убеждаемся, что колонки совпадают
column_names = X.columns
X = X[column_names]
X_test = X_test[column_names]

# Разделение на train/val (опционально, если хотите валидационную выборку)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)

train: (4988, 18) val: (1248, 18) test: (2498, 18)


## 3) Настройка эксперимента и запуск run

In [25]:
with mlflow.start_run(run_name="baseline_models", nested=True):
    # Baseline 1: Dummy (mean)
    dummy = DummyRegressor(strategy='mean')
    dummy.fit(X_train, y_train)
    y_test_dummy = dummy.predict(X_test)
    
    mlflow.log_metric("baseline_dummy_r2", r2_score(y_test, y_test_dummy))
    mlflow.log_metric("baseline_dummy_mae", mean_absolute_error(y_test, y_test_dummy))
    
    # Baseline 2: Linear Regression
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    y_test_lr = lr.predict(X_test)
    
    mlflow.log_metric("baseline_lr_r2", r2_score(y_test, y_test_lr))
    mlflow.log_metric("baseline_lr_mae", mean_absolute_error(y_test, y_test_lr))
    
    print("Baseline R2 scores:")
    print(f"  Dummy: {r2_score(y_test, y_test_dummy):.4f}")
    print(f"  Linear Regression: {r2_score(y_test, y_test_lr):.4f}")

2026/05/29 22:43:20 INFO mlflow.tracking._tracking_service.client: 🏃 View run baseline_models at: http://localhost:5050/#/experiments/1/runs/4fe6b584620344bd974424fb297681d1.
2026/05/29 22:43:20 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5050/#/experiments/1.


Baseline R2 scores:
  Dummy: -0.0001
  Linear Regression: 0.3744


In [26]:
experiment_name = "gradient_boosting_regression"
artifact_location = "s3://mlflow-bucket/mlflow"
client = MlflowClient()

exp = mlflow.get_experiment_by_name(experiment_name)
if exp is None:
    exp_id = client.create_experiment(
        name=experiment_name,
        artifact_location=artifact_location
    )
    print("Created experiment:", exp_id)
else:
    exp_id = exp.experiment_id
    print("Using existing experiment:", exp_id, "artifact_location=", exp.artifact_location)

mlflow.set_experiment(experiment_name)

registered_model_name = "gradient_boosting_model"

params = {
    "n_estimators": 100,
    "learning_rate": 0.1,
    "max_depth": 3,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "random_state": 42,
    "loss": "squared_error",
}

Using existing experiment: 1 artifact_location= s3://mlflow-bucket/mlflow


In [27]:
with mlflow.start_run(experiment_id=exp_id):
    print("Artifact URI for this run:", mlflow.get_artifact_uri())
    gbr = GradientBoostingRegressor(**params)
    gbr.fit(X_train, y_train)

    y_val_pred = gbr.predict(X_val)
    y_test_pred = gbr.predict(X_test)

    # Метрики на валидации
    val_metrics = {
        "val_mse": float(mean_squared_error(y_val, y_val_pred)),
        "val_mae": float(mean_absolute_error(y_val, y_val_pred)),
        "val_mape": float(mean_absolute_percentage_error(y_val, y_val_pred)),
        "val_r2": float(r2_score(y_val, y_val_pred)),
    }
    
    # Метрики на тесте
    test_metrics = {
        "test_mse": float(mean_squared_error(y_test, y_test_pred)),
        "test_mae": float(mean_absolute_error(y_test, y_test_pred)),
        "test_mape": float(mean_absolute_percentage_error(y_test, y_test_pred)),
        "test_r2": float(r2_score(y_test, y_test_pred)),
    }

    # Логируем параметры и метрики
    mlflow.log_params(params)
    mlflow.log_metrics(val_metrics)
    mlflow.log_metrics(test_metrics)

    # Выводим результаты
    print("\n=== Validation Metrics ===")
    for metric, value in val_metrics.items():
        print(f"{metric}: {value:.4f}")
    
    print("\n=== Test Metrics ===")
    for metric, value in test_metrics.items():
        print(f"{metric}: {value:.4f}")

    signature = infer_signature(X_train, gbr.predict(X_train))

    # Логируем модель
    model_info = mlflow.sklearn.log_model(
        sk_model=gbr,
        artifact_path="model",
        signature=signature,
        input_example=X_train.head(5),
        registered_model_name=registered_model_name,
    )

    # Логируем датасет
    mlflow.log_input(
        mlflow.data.from_pandas(df, source="train_with_new_features.csv"),
        context="training",
    )

    # Сохраняем важность признаков как артефакт
    feature_importance = pd.DataFrame({
        'feature': column_names,
        'importance': gbr.feature_importances_
    }).sort_values('importance', ascending=False)
    
    feature_importance.to_csv("feature_importance.csv", index=False)
    mlflow.log_artifact("feature_importance.csv")

    # Маркируем текущую версию как PRD в реестре моделей
    client = MlflowClient()
    new_version = model_info.registered_model_version
    client.set_model_version_tag(registered_model_name, new_version, "env", "PRD")
    client.set_registered_model_alias(registered_model_name, "prd", new_version)

    run_id = mlflow.active_run().info.run_id
    print(f"Run ID: {run_id}")
    print(f"Registered model version: {new_version}")
    print(f"Alias 'prd' points to version: {new_version}")
    print(f"Best R^2 on test: {test_metrics['test_r2']:.4f}")

Artifact URI for this run: s3://mlflow-bucket/mlflow/91e4805825264d98b1de9c0d45ed748f/artifacts

=== Validation Metrics ===
val_mse: 0.0021
val_mae: 0.0369
val_mape: 2.0364
val_r2: 0.7177

=== Test Metrics ===
test_mse: 0.0002
test_mae: 0.0111
test_mape: 5.7361
test_r2: 0.9578


Registered model 'gradient_boosting_model' already exists. Creating a new version of this model...
2026/05/29 22:44:09 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: gradient_boosting_model, version 3
Created version '3' of model 'gradient_boosting_model'.
2026/05/29 22:44:09 INFO mlflow.tracking._tracking_service.client: 🏃 View run merciful-hen-109 at: http://localhost:5050/#/experiments/1/runs/91e4805825264d98b1de9c0d45ed748f.
2026/05/29 22:44:09 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5050/#/experiments/1.


Run ID: 91e4805825264d98b1de9c0d45ed748f
Registered model version: 3
Alias 'prd' points to version: 3
Best R^2 on test: 0.9578


In [33]:
# Анализ ошибок
error_df = pd.DataFrame({
    'actual': y_test.values,
    'predicted': y_test_pred,
    'error': y_test.values - y_test_pred,
    'abs_error': np.abs(y_test.values - y_test_pred)
})

# Топ-20 ошибок
top_errors = error_df.nlargest(20, 'abs_error')
top_errors.to_csv("top_20_errors.csv", index=False)
mlflow.log_artifact("top_20_errors.csv")

print("Топ-20 ошибок:")
print(top_errors[['actual', 'predicted', 'error']].to_string())

under = top_errors[top_errors['error'] > 0]
over = top_errors[top_errors['error'] < 0]

print()
print(f"- Заниженные предсказания: {len(under)} шт")
print(f"- Завышенные предсказания: {len(over)} шт")

Топ-20 ошибок:
        actual  predicted     error
403   0.139755   0.236472 -0.096716
593   0.006926   0.092169 -0.085243
39    0.008161   0.090636 -0.082475
1545 -0.015550  -0.082913  0.067363
343  -0.019331  -0.079091  0.059760
2194  0.109838   0.167131 -0.057293
2213  0.086121   0.143102 -0.056981
1663  0.010765  -0.041886  0.052651
1623  0.014713  -0.036164  0.050877
2183 -0.030001  -0.078784  0.048784
1200  0.192578   0.145311  0.047267
1957  0.194223   0.149699  0.044524
353   0.140730   0.098458  0.042272
1495  0.176208   0.134489  0.041719
207   0.166997   0.126399  0.040597
1322  0.166997   0.126399  0.040597
294   0.009316  -0.031086  0.040401
2247  0.003050   0.043255 -0.040205
797   0.105508   0.145631 -0.040123
75    0.000756  -0.038819  0.039575

- Заниженные предсказания: 13 шт
- Завышенные предсказания: 7 шт


In [34]:
with mlflow.start_run(run_name="robustness_test", nested=True):
    noise_levels = [0.01, 0.05, 0.1]
    X_test_sample = X_test.iloc[:100]
    y_test_sample = y_test.iloc[:100]
    
    original_pred = gbr.predict(X_test_sample)
    original_mse = mean_squared_error(y_test_sample, original_pred)
    mlflow.log_metric("baseline_mse", original_mse)
    
    for noise in noise_levels:
        X_noisy = X_test_sample.copy()
        numeric_cols = X_noisy.select_dtypes(include=[np.number]).columns
        
        for col in numeric_cols:
            noise_vals = np.random.normal(0, noise * X_noisy[col].std(), len(X_noisy))
            X_noisy[col] = X_noisy[col] + noise_vals
        
        noisy_pred = gbr.predict(X_noisy)
        noisy_mse = mean_squared_error(y_test_sample, noisy_pred)
        change_pct = ((noisy_mse - original_mse) / original_mse) * 100
        
        mlflow.log_metric(f"mse_noise_{int(noise*100)}", noisy_mse)
        mlflow.log_metric(f"mse_change_{int(noise*100)}", change_pct)
        
        print(f"Noise {noise*100}%: MSE change = {change_pct:+.1f}%")

2026/05/29 22:46:00 INFO mlflow.tracking._tracking_service.client: 🏃 View run robustness_test at: http://localhost:5050/#/experiments/1/runs/ad5df9ff126446e3bf3b2f245ed88529.
2026/05/29 22:46:00 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5050/#/experiments/1.


Noise 1.0%: MSE change = +2.3%
Noise 5.0%: MSE change = -8.8%
Noise 10.0%: MSE change = +40.5%


In [36]:
train_sizes, train_scores, val_scores = learning_curve(
    gbr, X_train, y_train, cv=3, 
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='r2', n_jobs=-1
)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Training')
plt.plot(train_sizes, val_scores.mean(axis=1), 'o-', label='Validation')
plt.xlabel('Training examples')
plt.ylabel('R2 Score')
plt.title('Learning Curves')
plt.legend()
plt.grid(True)
plt.savefig('learning_curves.png')
mlflow.log_artifact('learning_curves.png')
plt.close()

## 4) Проверка результатов в MLflow

In [37]:
runs_df = mlflow.search_runs(
    experiment_names=[experiment_name],
    order_by=["metrics.test_r2 DESC"],
)

print(runs_df[["run_id", "metrics.test_r2", "metrics.test_mae", "artifact_uri"]].head())

                             run_id  metrics.test_r2  metrics.test_mae  \
0  91e4805825264d98b1de9c0d45ed748f         0.957801          0.011065   
1  20b61cb1b0434140a4273f212d091c01         0.957801          0.011065   
2  65a49d779f164bbeb084adc24d4c40ba         0.957801          0.011065   
3  1b4e875094544234b48b99bfd3719086         0.957801          0.011065   
4  ad5df9ff126446e3bf3b2f245ed88529              NaN               NaN   

                                        artifact_uri  
0  s3://mlflow-bucket/mlflow/91e4805825264d98b1de...  
1  s3://mlflow-bucket/mlflow/20b61cb1b0434140a427...  
2  s3://mlflow-bucket/mlflow/65a49d779f164bbeb084...  
3  s3://mlflow-bucket/mlflow/1b4e875094544234b48b...  
4  s3://mlflow-bucket/mlflow/ad5df9ff126446e3bf3b...  


In [38]:
# Показать запуски
print(mlflow.search_runs(experiment_names=[experiment_name]))

                             run_id experiment_id    status  \
0  ad5df9ff126446e3bf3b2f245ed88529             1  FINISHED   
1  ce47cf70c4c84750afaa2b0e3169a6a5             1   RUNNING   
2  91e4805825264d98b1de9c0d45ed748f             1  FINISHED   
3  4fe6b584620344bd974424fb297681d1             1  FINISHED   
4  20b50b1e0b2d40e9b9b10ba8866c1ff8             1    FAILED   
5  20b61cb1b0434140a4273f212d091c01             1  FINISHED   
6  65a49d779f164bbeb084adc24d4c40ba             1  FINISHED   
7  1b4e875094544234b48b99bfd3719086             1    FAILED   

                                        artifact_uri  \
0  s3://mlflow-bucket/mlflow/ad5df9ff126446e3bf3b...   
1  s3://mlflow-bucket/mlflow/ce47cf70c4c84750afaa...   
2  s3://mlflow-bucket/mlflow/91e4805825264d98b1de...   
3  s3://mlflow-bucket/mlflow/4fe6b584620344bd9744...   
4  s3://mlflow-bucket/mlflow/20b50b1e0b2d40e9b9b1...   
5  s3://mlflow-bucket/mlflow/20b61cb1b0434140a427...   
6  s3://mlflow-bucket/mlflow/65a49d779f1

## 5) Загрузка модели из Model Registry по тегу PRD (alias `prd`)

In [39]:
# Загрузка модели
loaded_model = mlflow.pyfunc.load_model(f"models:/{registered_model_name}@prd")

# Берем тестовые данные
test_sample = X_test.head(3).copy()

int_cols = ['schools_nearby', 'supermarket_nearby', 'mall_nearby', 'bar_nearby', 
            'cafe_nearby', 'restaurant_nearby', 'police_nearby', 'post_office_nearby',
            'place_of_worship_nearby', 'university_nearby', 'cinema_nearby', 
            'casino_nearby', 'nightclub_nearby']

float_cols = ['atm_group', 'lat', 'long', 'atm_nearby', 'population']

test_sample[int_cols] = test_sample[int_cols].astype('int64')
test_sample[float_cols] = test_sample[float_cols].astype('float64')

# Предсказание
sample_pred = loaded_model.predict(test_sample)
sample_pred

array([-0.04858019, -0.02344889, -0.01703288])